# Fourier Transform Demo: Periodic Neural Dynamics

This notebook walks through a simple example of applying the Fourier transform to calcium imaging data.
The goal is to show how stimulus-driven neuronal responses can be identified at a stimulus-locked frequency, along with brief analysis of the result.

The dataset in `/data` serves as an example; in practice, the same pipeline can be applied to other experimental recordings with slight variation.

The calcium imaging data comes from zebrafish performing phototaxis, the tendency of zebrafish to orient toward illumination. Phototaxis behavior was elicited in this dataset by a periodic half-field dark stimulus (more detail can be found in [original Neuron paper](<https://www.cell.com/neuron/pdf/S0896-6273(18)30844-4.pdf>)). The periodicity of the stimulus makes it suitable for Fourier analysis.


# Setup Instructions

This notebook is self-contained and will help you set up everything needed.

**How to use:**
1. Create a project folder on your computer, with:
   - a `data/` folder for datasets
   - a `notebooks/` folder for this notebook
2. Open a terminal in the project folder.
3. Create and activate a virtual environment.
4. Install dependencies.
5. Launch Jupyter Notebook and open this file.

No prior setup is required beyond having Python installed.


In [ ]:
# Step 1) Navigate to the project folder
cd path\to\your\project

# Step 2) Create a new environment named venv
python -m venv venv

# Step 3) Activate the environment
# PowerShell:
.\venv\Scripts\Activate.ps1
# Or in Command Prompt (cmd.exe):
venv\Scripts\activate.bat

# Step 4) Install the required dependencies
pip install -r requirements.txt

# Step 5) Launch Jupyter Notebook
python -m jupyter notebook


### Next steps inside Jupyter
- In the browser window that opens, navigate to the `notebooks/` folder.
- Open `Notebook_01.ipynb`.
- Ensure the kernel is set to the environment you just created (look for `(venv)` in the kernel name).
- Run the cells step by step to reproduce the analysis.

The next block fetch the directory, please check if the path for Project roots and Data folder are correct

In [ ]:
# Import required dependencies
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import h5py
import scipy

# Get the project root (assume the notebook lives inside /notebooks)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Define data folder
DATA_DIR = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("Data folder:", DATA_DIR)

# Check that the folder exists
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"❌ Data folder not found at {DATA_DIR}. "
        "Please make sure you have a 'data' folder in your project root."
    )


# Step 1: Identifying Stimulus Frequency with the Fourier Transform

In this section, we analyze the **phototaxis stimulus pattern** using the **Discrete Fourier Transform (DFT)**.
Since the stimulus was designed to be periodic, the Fourier transform allows us to extract key properties such as **frequency (or wavelength), phase, and amplitude** of the signal.

Although the stimulus here is artificially generated (and thus its properties are already known by design), this part serves as an **introduction to the Fourier transform**.  We will also define the core terminologies that will be used in later sections.

We will begin with loading the stimulus data. The data can be retrieved in /data/stimulus


In [ ]:
# read the stimulus data
stimulus = pd.read_excel(
    DATA_DIR / "stimulus.xlsx",
    header=None
).to_numpy()[0]

# Represent left/right stimulus by box-car functions
stimulus_right = [1 if x == 1 else 0 for x in stimulus]
stimulus_left = [1 if x == 2 else 0 for x in stimulus]

- The arrays above represent the **left- and right-sided phototaxis stimuli** as boxcar functions.
  - A **right-sided phototaxis stimulus** means the **left side of the fish is dark** while the **right side is illuminated**, driving the fish to orient toward the right.
  - Conversely, a **left-sided phototaxis stimulus** means the right side is dark and the left side is illuminated.
- Each array element corresponds to the **stimulus status at a given frame** (sampling rate ≈ 1.97 Hz).
  - A value of **1** indicates the stimulus was present.
  - A value of **0** indicates the stimulus was absent.
  - Because the sampling rate is not an exact integer, we use **frame index** (rather than seconds) as the time unit for analysis.
- To visualize the stimulus pattern for the first 500 frames, run the following code cell.


In [ ]:
plt.figure(figsize=(10,2))
plt.plot(stimulus_left[:500],c = "red")
plt.plot(stimulus_right[:500],c = "blue")
plt.xlabel("Frame")
plt.legend(["left", "right"])
plt.show()